# Introduction

## Problem Statement



## Section 1:  Work environment set up

In [1]:
# --- Standard Imports ---
import pandas as pd
import numpy as np
import warnings
import logging
import matplotlib.pyplot as plt
import seaborn as sns
import re
import sys



from pathlib import Path
from functools import reduce
from sklearn.preprocessing import MinMaxScaler
from collections import Counter

# --- Setup ---
sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
warnings.filterwarnings('ignore')

# --- Paths ---
data_dir = Path("data")
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)
figures_dir = output_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

log_dir = Path("logs")
log_dir.mkdir(exist_ok=True)

image_dir = Path("outputs/images")
image_dir.mkdir(parents=True, exist_ok=True)

# Clear existing handlers
logging.getLogger().handlers.clear()

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_dir / "project.log", mode='w', encoding='utf-8'),
        logging.StreamHandler(sys.stdout)
    ]
)


print("Environment ready. Paths and logging configured.")



Environment ready. Paths and logging configured.


# Section 2: Load and prepare Data sets

## Section 2.1: Load utility functions and national datasets ----

In [3]:
# Section 2A: Load utility functions and national datasets ----

# Import custom utility functions
from utils import (
    standardize_column_names,
    extract_key_indicators,
    filter_to_county_level,
    log_duplicate_attributes,
    clean_and_extract_year,
    nca_counties
)

data_dir = Path("data")
complete_dir = data_dir / "complete_sets"

complete_files = {
    "edu": "Education2023.csv",
    "pop": "PopulationEstimates.csv",
    "poverty": "Poverty2023.csv",
    "unemp": "Unemployment2023.csv"
}


# Load, clean, and extract year from each dataset
complete_data = {}

for key, filename in complete_files.items():
    path = complete_dir / filename
    try:
        df = pd.read_csv(path, encoding='cp1252')
        df = standardize_column_names(df)
        logging.info(f" {key} columns after cleaning: {df.columns.tolist()}")
        df = filter_to_county_level(df)
        df = clean_and_extract_year(df)

        # Rename fields if necessary
        df.rename(columns={'area_name': 'county'}, inplace=True)

        log_duplicate_attributes(df, key)
        complete_data[key] = df
        logging.info(f"Loaded {filename}: {df.shape[0]} rows")

    except Exception as e:
        logging.error(f" Failed to load {filename}: {e}")


# Confirmation message
logging.info("Utility functions from utils.py loaded successfully.")


2025-07-10 09:11:04,477 [INFO]  edu columns after cleaning: ['ïfips_code', 'state', 'area_name', 'attribute', 'value']
2025-07-10 09:11:04,972 [WARNING] edu has duplicate county-attribute pairs.
2025-07-10 09:11:04,974 [INFO] Loaded Education2023.csv: 169245 rows
2025-07-10 09:11:05,081 [INFO]  pop columns after cleaning: ['fipstxt', 'state', 'area_name', 'attribute', 'value']
2025-07-10 09:11:05,550 [WARNING] pop has duplicate county-attribute pairs.
2025-07-10 09:11:05,551 [INFO] Loaded PopulationEstimates.csv: 205108 rows
2025-07-10 09:11:05,595 [INFO]  poverty columns after cleaning: ['ïfips_code', 'state', 'area_name', 'attribute', 'value']
2025-07-10 09:11:05,775 [WARNING] poverty has duplicate county-attribute pairs.
2025-07-10 09:11:05,775 [INFO] Loaded Poverty2023.csv: 79961 rows
2025-07-10 09:11:05,916 [INFO]  unemp columns after cleaning: ['fips_code', 'state', 'area_name', 'attribute', 'value']
2025-07-10 09:11:06,652 [INFO] Loaded Unemployment2023.csv: 324636 rows
2025-07-

## Section 2.2: Load and process national datasets

In [4]:
# Section 2.2: Load and process national datasets

complete_dir = data_dir / "complete_sets"
complete_files = {
    "edu": "Education2023.csv",
    "pop": "PopulationEstimates.csv",
    "poverty": "Poverty2023.csv",
    "unemp": "Unemployment2023.csv"
}

state_lookup = None

# Choose your analysis year here
selected_year = '2023'

for key, filename in complete_files.items():
    path = complete_dir / filename
    try:
        # Load and standardize
        df = pd.read_csv(path, encoding='cp1252')
        df = standardize_column_names(df)
        df = filter_to_county_level(df)
        df.rename(columns={'area_name': 'county', 'fips_code': 'fips', 'fipstxt': 'fips'}, inplace=True)
        df['county'] = df['county'].str.strip().str.lower()

        # Extract year from attribute
        if 'attribute' in df.columns:
            df = clean_and_extract_year(df)

            # Only filter datasets where year tagging applies
            if key in ['pop', 'poverty', 'unemp']:
                df = df[df['attribute_year'] == selected_year]

        # Save state info once from education
        if key == 'edu':
            state_lookup = df[['county', 'state']].drop_duplicates().copy()
            state_lookup['county'] = state_lookup['county'].str.lower().str.strip()
            state_lookup['state'] = state_lookup['state'].str.upper().str.strip()

        log_duplicate_attributes(df, key)
        complete_data[key] = df
        logging.info(f" Loaded and processed {filename} with year filter: {selected_year if key != 'edu' else 'N/A'}")

    except Exception as e:
        logging.error(f" Failed to load {filename}: {e}")


2025-07-10 09:11:13,524 [WARNING] edu has duplicate county-attribute pairs.
2025-07-10 09:11:13,529 [INFO]  Loaded and processed Education2023.csv with year filter: N/A
2025-07-10 09:11:14,103 [WARNING] pop has duplicate county-attribute pairs.
2025-07-10 09:11:14,107 [INFO]  Loaded and processed PopulationEstimates.csv with year filter: 2023
2025-07-10 09:11:14,278 [WARNING] poverty has duplicate county-attribute pairs.
2025-07-10 09:11:14,280 [INFO]  Loaded and processed Poverty2023.csv with year filter: 2023
2025-07-10 09:11:15,128 [INFO]  Loaded and processed Unemployment2023.csv with year filter: 2023


### Section 2.3: Discover Common Year Across Datasets

In [5]:
# Section 2.3: Discover Common Year Across Datasets

def get_attribute_year_counts(df):
    """Counts how often each extracted attribute_year appears."""
    if 'attribute' in df.columns:
        df = clean_and_extract_year(df)
        return Counter(df['attribute_year'].dropna().astype(str))
    return {}

year_summary = {}

# Loop through each dataset and extract year counts
for key, df in complete_data.items():
    year_counts = get_attribute_year_counts(df)
    year_summary[key] = year_counts

# Display results
print("Year coverage by dataset:")
all_years = set()

for key, counts in year_summary.items():
    print(f"\n {key.upper()}:")
    if counts:
        for year, count in sorted(counts.items()):
            print(f"  {year}: {count}")
            all_years.add(year)
    else:
        print("  No attribute_year values found.")

# Find common years across all datasets that have year values
datasets_with_years = [set(c.keys()) for c in year_summary.values() if c]
common_years = set.intersection(*datasets_with_years) if datasets_with_years else set()



print("\n Common years across all datasets with usable attribute_year:", sorted(common_years))


Year coverage by dataset:

 EDU:
  1970: 25488
  1980: 26136
  1990: 26167
  2000: 26176
  2012: 26192
  2013: 6442
  2023: 29422
  2024: 3222

 POP:
  2023: 53619

 POVERTY:
  2023: 19223

 UNEMP:
  2023: 19326

 Common years across all datasets with usable attribute_year: ['2023']


## Section 2.4: Subset Arkansas and NCA Counties (No Full Merge)

In [6]:
# Section 2.4: Subset Arkansas and NCA Counties (No Full Merge)

# Define NCA counties (lowercase, cleaned)
nca_cleaned = [c.lower().strip() for c in nca_counties]

# Function to filter, clean, and subset each dataset
def get_nca_subset(df, dataset_name):
    df = df.copy()
    df = df[df['state'].str.upper() == 'AR']
    df['county'] = (
        df['county']
        .str.replace(" county", "", regex=False)
        .str.replace(", ar", "", regex=False)
        .str.strip()
        .str.lower()
    )
    df_nca = df[df['county'].isin(nca_cleaned)].copy()
    logging.info(f"{dataset_name.upper()} → NCA rows: {df_nca.shape[0]}")
    return df_nca

# Get cleaned NCA subsets
edu_nca = get_nca_subset(complete_data['edu'], 'edu')
poverty_nca = get_nca_subset(complete_data['poverty'], 'poverty')
unemp_nca = get_nca_subset(complete_data['unemp'], 'unemp')
pop_nca = get_nca_subset(complete_data['pop'], 'pop')

# Pivot each to wide format
edu_wide = edu_nca.pivot_table(index='county', columns='attribute', values='value', aggfunc='first')
poverty_wide = poverty_nca.pivot_table(index='county', columns='attribute', values='value', aggfunc='first')
unemp_wide = unemp_nca.pivot_table(index='county', columns='attribute', values='value', aggfunc='first')
pop_wide = pop_nca.pivot_table(index='county', columns='attribute', values='value', aggfunc='first')

# Merge NCA-wide datasets
df_nca = reduce(
    lambda left, right: pd.merge(left, right, on='county', how='outer'),
    [edu_wide, poverty_wide, unemp_wide, pop_wide]
)
# Flatten MultiIndex columns (from pivot operations)
df_nca.columns = [col if isinstance(col, str) else col[1] for col in df_nca.columns]

# Save and log
df_nca.to_csv(output_dir / f"nca_dataset_cleaned_{selected_year}.csv")
logging.info(f"NCA final dataset saved: nca_dataset_cleaned_{selected_year}.csv")
print(f"NCA dataset ready with shape: {df_nca.shape}")

2025-07-10 09:11:24,235 [INFO] EDU → NCA rows: 676
2025-07-10 09:11:24,241 [INFO] POVERTY → NCA rows: 78
2025-07-10 09:11:24,247 [INFO] UNEMP → NCA rows: 78
2025-07-10 09:11:24,278 [INFO] POP → NCA rows: 221
2025-07-10 09:11:24,305 [INFO] NCA final dataset saved: nca_dataset_cleaned_2023.csv
NCA dataset ready with shape: (13, 81)


## Section 2.5 Streamline NCA data set 

In [ ]:
# Section 2.5 Streamline NCA data set 

print("🔎 Columns BEFORE renaming:")
print(df_nca.columns.tolist())

# Ensure 'county' is a column, not just an index
df_nca = df_nca.reset_index()

# Rename long education column labels for clarity
df_nca = df_nca.rename(columns={
    "percent of adults who are high school graduates (or equivalent), 2019-23": "HighSchoolGradRate",
    "percent of adults with a bachelor's degree or higher, 2019-23": "BachelorsDegreeRate",
    "pctpovall_2023": "PovertyRate",
    "unemployment_rate_2023": "UnemploymentRate",
    "pop_estimate_2023": "Population"
})


# If the education rates are actually raw counts (verify first!)
df_nca['BachelorsDegreePct'] = (df_nca['BachelorsDegreeRate'] / df_nca['Population']) * 100
df_nca['HighSchoolGradPct'] = (df_nca['HighSchoolGradRate'] / df_nca['Population']) * 100


summary_vars = [
    'BachelorsDegreePct',
    'HighSchoolGradPct',
    'PovertyRate',
    'UnemploymentRate',
    'Population'
]

# Select only the summary variables and 'county'
summary_vars = [var for var in summary_vars if var in df_nca.columns]
df_nca = df_nca[summary_vars + ['county']].set_index('county')
# Log final columns
logging.info(f"Final NCA columns: {df_nca.columns.tolist()}")


# Save final NCA dataset
df_nca.to_csv(output_dir / "nca_final.csv", index=False)
logging.info("NCA final dataset saved: nca_final.csv")

🔎 Columns BEFORE renaming:
['2013 rural-urban continuum code', '2013 urban influence code', '2023 rural-urban continuum code', '2024 urban influence code', "bachelor's degree or higher, 1990", "bachelor's degree or higher, 2000", "bachelor's degree or higher, 2008-12", "bachelor's degree or higher, 2019-23", 'four years of college or higher, 1970', 'four years of college or higher, 1980', 'high school diploma only, 1970', 'high school diploma only, 1980', 'high school graduate (or equivalency), 1990', 'high school graduate (or equivalency), 2000', 'high school graduate (or equivalency), 2008-12', 'high school graduate (or equivalency), 2019-23', 'less than a high school diploma, 1970', 'less than a high school diploma, 1980', 'less than high school graduate, 1990', 'less than high school graduate, 2000', 'less than high school graduate, 2008-12', 'less than high school graduate, 2019-23', 'percent of adults completing four years of college or higher, 1970', 'percent of adults completin

# Section 3 Exploratory Data Overview (Tables & Summary Insights)

## Section 3.1 Dataset Overview

In [ ]:
# Section 3.1: Dataset Overview
print("Dataset Dimensions (rows, columns):", df_nca_final.shape)

print("\n Column Names:")
for col in df_nca.columns:
    print(" -", col)

print("\n Dataset Info:")
df_nca.info()

print("\n Preview of First 5 Rows:")
display(df_nca.head())

NameError: name 'df_nca_final' is not defined

## Section 3.2 Data Dictionary 

In [9]:
# Section 3.3: Data Dictionary Summary

# Manually define the variable descriptions
data_dictionary = {
    "BachelorsDegreeRate": "Raw number of residents with a bachelor’s degree",
    "HighSchoolGradRate": "Raw number of residents with a high school diploma or higher",
    "Population": "Estimated total population of the county",
    "PovertyRate": "Percentage of residents living below the poverty line",
    "UnemploymentRate": "Percentage of labor force unemployed",
    "BachelorsDegreePct": "% of population with a bachelor’s degree (calculated)",
    "HighSchoolGradPct": "% of population with a high school diploma (calculated)"
}

# Convert to a DataFrame for display and export
dictionary_df = pd.DataFrame.from_dict(data_dictionary, orient='index', columns=['Description'])
dictionary_df.index.name = 'Variable'

# Display and save
display(dictionary_df)

dictionary_df.to_csv(output_dir / "nca_data_dictionary.csv")
logging.info("Data dictionary saved to: nca_data_dictionary.csv")


,Description
Variable,
BachelorsDegreeRate,Raw number of residents with a bachelor’s degree
HighSchoolGradRate,Raw number of residents with a high school dip...
Population,Estimated total population of the county
PovertyRate,Percentage of residents living below the pover...
UnemploymentRate,Percentage of labor force unemployed
BachelorsDegreePct,% of population with a bachelor’s degree (calc...
HighSchoolGradPct,% of population with a high school diploma (ca...


2025-07-10 09:12:02,682 [INFO] Data dictionary saved to: nca_data_dictionary.csv


## Section 3.3 Descriptive Statistics

In [10]:
# Section 3.3: Per-Variable County-Level Tables

percent_vars = [
    'BachelorsDegreePct',
    'HighSchoolGradPct',
    'PovertyRate',
    'UnemploymentRate'
]

for var in percent_vars:
    var_table = df_nca[[var]].copy()
    var_table.columns = ['Value']
    var_table.index.name = 'County'

    # Display
    print(f"\n📊 {var} by County:")
    display(var_table)

    # Save to CSV
    filename = f"section3_2_{var.lower()}_by_county.csv"
    var_table.to_csv(figures_dir / filename)
    logging.info(f"[Section 3.2] Saved: {filename}")



📊 BachelorsDegreePct by County:


,Value
County,
baxter,0.043729
cleburne,0.071801
fulton,0.126570
independence,0.044027
izard,0.132050
jackson,0.078980
marion,0.088152
searcy,0.175360
sharp,0.066387


2025-07-10 09:12:50,756 [INFO] [Section 3.2] Saved: section3_2_bachelorsdegreepct_by_county.csv

📊 HighSchoolGradPct by County:


,Value
County,
baxter,0.080813
cleburne,0.155012
fulton,0.328741
independence,0.102839
izard,0.251546
jackson,0.255850
marion,0.206390
searcy,0.467404
sharp,0.225743


2025-07-10 09:12:50,763 [INFO] [Section 3.2] Saved: section3_2_highschoolgradpct_by_county.csv

📊 PovertyRate by County:


,Value
County,
baxter,16.0
cleburne,13.5
fulton,17.5
independence,13.5
izard,20.3
jackson,24.6
marion,17.4
searcy,20.2
sharp,19.7


2025-07-10 09:12:50,769 [INFO] [Section 3.2] Saved: section3_2_povertyrate_by_county.csv

📊 UnemploymentRate by County:


,Value
County,
baxter,3.6
cleburne,3.9
fulton,3.8
independence,3.4
izard,5.1
jackson,5.1
marion,4.1
searcy,4.3
sharp,3.9


2025-07-10 09:12:50,775 [INFO] [Section 3.2] Saved: section3_2_unemploymentrate_by_county.csv


## Section 3.4 Outlier/Range Table

In [11]:
# Section 3.5: Outlier / Range Table

print(" Outlier and Range Summary for Selected Indicators:\n")

for var in summary_vars:
    print(f" Top 3 counties by {var}:")
    display(df_nca[[var]].sort_values(by=var, ascending=False).head(3))
    
    print(f" Bottom 3 counties by {var}:")
    display(df_nca[[var]].sort_values(by=var, ascending=True).head(3))
    
    print("-" * 60)


 Outlier and Range Summary for Selected Indicators:

 Top 3 counties by BachelorsDegreePct:


,BachelorsDegreePct
county,
woodruff,0.266446
searcy,0.175360
izard,0.132050


 Bottom 3 counties by BachelorsDegreePct:


,BachelorsDegreePct
county,
white,0.025242
baxter,0.043729
independence,0.044027


------------------------------------------------------------
 Top 3 counties by HighSchoolGradPct:


...

------------------------------------------------------------
 Top 3 counties by Population:


,Population
county,
white,78452.0
baxter,42875.0
independence,38320.0


 Bottom 3 counties by Population:


,Population
county,
woodruff,5964.0
searcy,7806.0
fulton,12421.0


------------------------------------------------------------


## Section 3.5 County Summary Table

In [12]:
# Section 3.4: One CSV per County Profile

export_vars = percent_vars + ['Population']

for county, row in df_nca[export_vars].iterrows():
    county_df = row.to_frame(name='Value')
    county_df.index.name = 'Variable'
    
    filename = f"section3_4_{county.lower().replace(' ', '_')}_profile.csv"
    county_df.to_csv(figures_dir / filename)
    logging.info(f"Saved profile: {filename}")


2025-07-10 09:13:25,560 [INFO] Saved profile: section3_4_baxter_profile.csv
2025-07-10 09:13:25,561 [INFO] Saved profile: section3_4_cleburne_profile.csv
2025-07-10 09:13:25,563 [INFO] Saved profile: section3_4_fulton_profile.csv
2025-07-10 09:13:25,565 [INFO] Saved profile: section3_4_independence_profile.csv
2025-07-10 09:13:25,566 [INFO] Saved profile: section3_4_izard_profile.csv
2025-07-10 09:13:25,568 [INFO] Saved profile: section3_4_jackson_profile.csv
2025-07-10 09:13:25,569 [INFO] Saved profile: section3_4_marion_profile.csv
2025-07-10 09:13:25,570 [INFO] Saved profile: section3_4_searcy_profile.csv
2025-07-10 09:13:25,571 [INFO] Saved profile: section3_4_sharp_profile.csv
2025-07-10 09:13:25,572 [INFO] Saved profile: section3_4_stone_profile.csv
2025-07-10 09:13:25,573 [INFO] Saved profile: section3_4_van_buren_profile.csv
2025-07-10 09:13:25,575 [INFO] Saved profile: section3_4_white_profile.csv
2025-07-10 09:13:25,576 [INFO] Saved profile: section3_4_woodruff_profile.csv


# Section 4 Data Visualizations & Pattern Discovery

In [ ]:
## Section 3.2: Extract and Name Key Indicator Variables

#  Print all column names for inspection
print("\n All column names in df_nca:")
for col in df_nca_final.columns:
    print(col)

# 🔍 Extract key indicators
df_nca, used_columns, year = extract_key_indicators(df_nca, min_year=2022)

# Print matched column summary
print("Most common year in column names:", year)
print("Education columns used:", used_columns['education'])
print("Poverty columns used:", used_columns['poverty'])
print("Unemployment columns used:", used_columns['unemployment'])
print("Population columns used:", used_columns['population'])

# Compute education percentages (relative to population)
df_nca['BachelorsDegreePct'] = (df_nca['BachelorsDegreeRate'] / df_nca['Population']) * 100
df_nca['HighSchoolGradPct'] = (df_nca['HighSchoolGradRate'] / df_nca['Population']) * 100

# Define clean variable set for EDA (use percentages)
variables = ['BachelorsDegreePct', 'HighSchoolGradPct', 'PovertyRate', 'UnemploymentRate', 'Population']

# Preview the cleaned dataset
print("\n Preview of standardized indicators:")
display(df_nca[variables].head())

# Debug: Print all matched and available columns
print("\n Matched columns by indicator:")
for key, cols in used_columns.items():
    print(f"  - {key}: {cols}")

print("\nAll available columns:")
print(df_nca.columns.tolist())




 All column names in df_nca:
BachelorsDegreePct
HighSchoolGradPct
PovertyRate
UnemploymentRate
Population
2025-07-10 09:15:40,770 [INFO] Most common year in column names: None
2025-07-10 09:15:40,771 [INFO] Education columns: ['BachelorsDegreePct', 'HighSchoolGradPct']
2025-07-10 09:15:40,771 [INFO] Poverty columns: []
2025-07-10 09:15:40,772 [INFO] Unemployment columns: []
2025-07-10 09:15:40,772 [INFO] Population columns: []
Most common year in column names: None
Education columns used: ['BachelorsDegreePct', 'HighSchoolGradPct']
Poverty columns used: []
Unemployment columns used: []
Population columns used: []

 Preview of standardized indicators:


,BachelorsDegreePct,HighSchoolGradPct,PovertyRate,UnemploymentRate,Population
county,,,,,
baxter,NaN,NaN,NaN,NaN,NaN
cleburne,NaN,NaN,NaN,NaN,NaN
fulton,NaN,NaN,NaN,NaN,NaN
independence,NaN,NaN,NaN,NaN,NaN
izard,NaN,NaN,NaN,NaN,NaN



 Matched columns by indicator:
  - education: ['BachelorsDegreePct', 'HighSchoolGradPct']
  - poverty: []
  - unemployment: []
  - population: []

All available columns:
['BachelorsDegreePct', 'HighSchoolGradPct', 'PovertyRate', 'UnemploymentRate', 'Population', 'BachelorsDegreeRate', 'HighSchoolGradRate', 'Year']


In [ ]:
education_cols = used_columns['education'][:2]  # Bachelor's + High School

# Education distribution across counties
title = "Education Indicators by County"
df[education_cols].T.plot(kind='bar', figsize=(14, 6), title=title)
plt.ylabel("Percent or Count")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

# Standardize filename
filename = title.lower().replace(" ", "_").replace("’", "").replace("'", "")
plt.savefig(image_dir / f"{filename}.png", dpi=300, bbox_inches='tight')
logging.info(f" Saved: {filename}.png")

plt.show()



In [ ]:
variables = ['PovertyRate', 'UnemploymentRate', 'HighSchoolGradPct', 'BachelorsDegreePct']

for var in variables:
    plt.figure(figsize=(8, 4))
    
    title = f"Distribution of {var}"
    
    # Use the correct DataFrame
    sns.histplot(df_nca[var], kde=True, bins=20)
    
    plt.title(title)
    plt.xlabel(var)
    plt.ylabel('Frequency')
    plt.tight_layout()
    
    # Save with safe filename
    filename = title.lower().replace(" ", "_").replace("’", "").replace("'", "")
    plt.savefig(image_dir / f"{filename}.png", dpi=300, bbox_inches='tight')
    logging.info(f"📷 Saved: {filename}.png")
    
    plt.show()


